In [5]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")

In [6]:
file_path = "../data/raw/maize_yield_rwanda_rw.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of records:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Number of records: 1426
Number of columns: 20


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
0,RW-MZ-00001,2015,A,Eastern,Bugesera,1349,NaN,22.9,NaN,Clay,RHM104,Improved,117.8,57.4,1.41,61803,0.63,No,NaN,2.71
1,RW-MZ-00002,2015,A,Eastern,Bugesera,1427,489.8,22.1,6.02,Volcanic,RHM104,Improved,38.1,62.7,3.69,52919,0.17,No,0.339,4.09
2,RW-MZ-00003,2015,A,Northern,Burera,2033,722.6,16.3,5.73,Clay loam,Local,Improved,121.1,50.7,5.18,54406,0.22,No,0.049,2.58
3,RW-MZ-00004,2015,A,Northern,Burera,2026,794.4,16.4,5.20,Clay loam,ZM621,Improved,93.7,60.4,1.68,48119,0.48,No,0.171,2.45
4,RW-MZ-00005,2015,A,Northern,Gakenke,1974,644.2,18.3,5.92,Loam,Local,Improved,84.1,70.4,0.36,56787,1.16,No,0.001,2.09


In [7]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all"))

Dataset shape: (1426, 20)

Column names:
['record_id', 'year', 'season', 'province', 'district', 'altitude_m', 'rainfall_mm', 'avg_temp_c', 'soil_ph', 'soil_type', 'seed_variety', 'farming_practice', 'npk_kg_ha', 'urea_kg_ha', 'manure_t_ha', 'plant_density_per_ha', 'farm_area_ha', 'irrigation', 'pest_incidence_index', 'yield_t_ha']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1426 entries, 0 to 1425
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   record_id             1426 non-null   object 
 1   year                  1426 non-null   int64  
 2   season                1426 non-null   object 
 3   province              1426 non-null   object 
 4   district              1426 non-null   object 
 5   altitude_m            1426 non-null   int64  
 6   rainfall_mm           1415 non-null   float64
 7   avg_temp_c            1426 non-null   float64
 8   soil_ph               

,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
count,1426,1426.000000,1426,1426,1426,1426.000000,1415.000000,1426.000000,1412.000000,1426,1426,1426,1426.000000,1426.000000,1426.000000,1426.000000,1426.000000,1426,1412.000000,1426.000000
unique,1426,NaN,3,5,30,NaN,NaN,NaN,NaN,5,6,2,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN
top,RW-MZ-01426,NaN,A,Southern,Karongi,NaN,NaN,NaN,NaN,Loam,ZM621,Improved,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
freq,1,NaN,600,388,56,NaN,NaN,NaN,NaN,410,297,783,NaN,NaN,NaN,NaN,NaN,1298,NaN,NaN
mean,NaN,2019.506311,NaN,NaN,NaN,1704.321879,516.165371,19.404137,5.835156,NaN,NaN,NaN,66.991304,38.767602,2.083415,46322.309958,0.802104,NaN,0.283499,2.577454
std,NaN,2.855643,NaN,NaN,NaN,221.267632,155.676106,1.645486,0.443019,NaN,NaN,NaN,47.311529,31.571860,1.361537,9695.817375,0.511796,NaN,0.167093,0.700232
min,NaN,2015.000000,NaN,NaN,NaN,1243.000000,150.000000,15.400000,4.500000,NaN,NaN,NaN,0.000000,0.000000,0.000000,19337.000000,0.100000,NaN,0.001000,1.010000
25%,NaN,2017.000000,NaN,NaN,NaN,1507.000000,419.350000,18.025000,5.530000,NaN,NaN,NaN,26.300000,11.500000,1.060000,38471.500000,0.460000,NaN,0.153000,2.060000
50%,NaN,2020.000000,NaN,NaN,NaN,1702.500000,530.600000,19.400000,5.840000,NaN,NaN,NaN,62.450000,33.100000,2.005000,46901.000000,0.670000,NaN,0.261000,2.575000
75%,NaN,2022.000000,NaN,NaN,NaN,1879.750000,627.450000,20.700000,6.130000,NaN,NaN,NaN,104.975000,61.875000,3.027500,53806.250000,0.980000,NaN,0.390250,3.040000


In [8]:
data_dictionary = {
    "record_id": "Unique record identifier",
    "year": "Year of farming",
    "season": "Agricultural season",
    "province": "Province where farming occurred",
    "district": "District where farming occurred",
    "altitude_m": "Farm altitude in metres",
    "rainfall_mm": "Rainfall in millimetres",
    "avg_temp_c": "Average temperature in Celsius",
    "soil_ph": "Soil acidity level",
    "soil_type": "Type of soil",
    "seed_variety": "Maize seed variety",
    "farming_practice": "Farming method",
    "npk_kg_ha": "NPK fertilizer per hectare",
    "urea_kg_ha": "Urea fertilizer per hectare",
    "manure_t_ha": "Manure per hectare",
    "plant_density_per_ha": "Plants per hectare",
    "farm_area_ha": "Farm area in hectares",
    "irrigation": "Whether irrigation is used",
    "pest_incidence_index": "Pest incidence measurement",
    "yield_t_ha": "Maize yield in tonnes per hectare"
}

dictionary_df = pd.DataFrame(
    list(data_dictionary.items()),
    columns=["Column", "Description"]
)

display(dictionary_df)

dictionary_df.to_csv(
    "../docs/data_dictionary.csv",
    index=False
)

,Column,Description
0,record_id,Unique record identifier
1,year,Year of farming
2,season,Agricultural season
3,province,Province where farming occurred
4,district,District where farming occurred
5,altitude_m,Farm altitude in metres
6,rainfall_mm,Rainfall in millimetres
7,avg_temp_c,Average temperature in Celsius
8,soil_ph,Soil acidity level
9,soil_type,Type of soil
